# 1. Data Understanding

This dataset is historical and anonymized. `Time` is elapsed seconds, not a calendar timestamp.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.data_validation import load_transactions
df=load_transactions(ROOT/"data"/"creditcard.csv")
df.shape, df.dtypes
display(df.describe().T)

In [ ]:
print("Missing values:",int(df.isna().sum().sum()))
print("Duplicate rows beyond first copies:",int(df.duplicated().sum()))
print("Rows that belong to duplicate groups:",int(df.duplicated(keep=False).sum()))
print("Target counts:\n",df.Class.value_counts().sort_index())
print(f"Fraud prevalence: {df.Class.mean():.6%}")
from sklearn.model_selection import train_test_split
from src.train import group_split
from src.data_validation import duplicate_split_audit
X=df.drop(columns="Class"); y=df.Class
row_train,row_test=train_test_split(df.index,test_size=.2,stratify=y,random_state=42)
group_train,group_test,_,_=group_split(X,y)
print("Row-random split duplicate audit:",duplicate_split_audit(df,row_train,row_test))
print("Group split duplicate audit:",duplicate_split_audit(df,group_train.index,group_test.index))

Exact feature duplicates have no conflicting target labels in this data. The prior row-random split allowed duplicate groups to cross train/test; the corrected workflow keeps identical feature rows together. The full source retains duplicates, since they may represent repeated transactions and removal would change the evaluation population.